<a href="https://colab.research.google.com/github/davidgeraldi/Calculo-Numerico-UNIFESP/blob/main/Atividade_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Cálculo Numérico — Lista de Exercícios
Sistemas lineares e implementação numérica
"""

import numpy as np


# ---------------------------------------------------------------------------
# EXERCÍCIO 4 — Eliminação de Gauss SEM pivoteamento
# ---------------------------------------------------------------------------
def gauss_sem_pivoteamento(A, b, mostrar_passos=False):
    """
    Resolve A x = b por eliminação de Gauss sem pivoteamento.

    Parâmetros
    ----------
    A : matriz n x n dos coeficientes
    b : vetor independente de tamanho n
    mostrar_passos : se True, imprime a matriz aumentada a cada etapa

    Retorna
    -------
    x : vetor solução (np.ndarray)
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    # 1) Forma a matriz aumentada [A | b]
    M = np.hstack([A, b.reshape(-1, 1)])
    if mostrar_passos:
        print("Matriz aumentada inicial:\n", M, "\n")

    # 2) Eliminação: zera os elementos abaixo de cada pivô M[k, k]
    for k in range(n - 1):
        pivo = M[k, k]
        if abs(pivo) < 1e-12:
            # Sem pivoteamento, pivô nulo inviabiliza o método
            raise ZeroDivisionError(
                f"Pivô nulo na etapa {k + 1}: é necessário pivoteamento."
            )
        for i in range(k + 1, n):
            m = M[i, k] / pivo            # multiplicador m_ik
            M[i, k:] = M[i, k:] - m * M[k, k:]   # L_i <- L_i - m * L_k
            if mostrar_passos:
                print(f"L{i + 1} <- L{i + 1} - ({m:.4f}) * L{k + 1}")
        if mostrar_passos:
            print(M, "\n")

    # 3) Substituição regressiva (de baixo para cima)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - soma) / M[i, i]

    return x


# --- Teste com o sistema do exercício 2 ---
print("=" * 60)
print("EXERCÍCIO 4 — Gauss sem pivoteamento (sistema do exercício 2)")
print("=" * 60)

A2 = [[2, 1, -1],
      [-3, -1, 2],
      [-2, 1, 2]]
b2 = [8, -11, -3]

x_gauss = gauss_sem_pivoteamento(A2, b2, mostrar_passos=True)
x_numpy = np.linalg.solve(np.array(A2, dtype=float), np.array(b2, dtype=float))

print("Solução (Gauss implementado):", x_gauss)
print("Solução (numpy.linalg.solve):", x_numpy)
print("Diferença máxima |x_gauss - x_numpy|:", np.max(np.abs(x_gauss - x_numpy)))
print("Resíduo ||A x - b||:",
      np.linalg.norm(np.array(A2) @ x_gauss - np.array(b2)))

# INTERPRETAÇÃO:
# O método implementado retorna x = 2, y = 3, z = -1, idêntico (a menos de
# erro de arredondamento da ordem de 1e-16) ao resultado de numpy.linalg.solve,
# que usa fatoração LU com pivoteamento parcial. Como os pivôs deste sistema
# (2, 1/2, -1) são todos não nulos, o pivoteamento não era necessário aqui.
# Observação: sem pivoteamento o método falha quando algum pivô é zero e pode
# ser numericamente instável quando o pivô é muito pequeno.


# ---------------------------------------------------------------------------
# VERIFICAÇÃO DOS EXERCÍCIOS 1 E 3 (resolução manual está no documento)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXERCÍCIO 1 — forma matricial (verificação das soluções)")
print("=" * 60)

A1a = np.array([[2, 1], [1, -3]], dtype=float)
b1a = np.array([7, -5], dtype=float)
print("1a) A =\n", A1a, "\n    b =", b1a, "\n    solução:", np.linalg.solve(A1a, b1a))

A1b = np.array([[1, 2, -1], [3, -1, 2], [2, 1, 1]], dtype=float)
b1b = np.array([2, 9, 7], dtype=float)
print("1b) A =\n", A1b, "\n    b =", b1b, "\n    solução:", np.linalg.solve(A1b, b1b))

print("\n" + "=" * 60)
print("EXERCÍCIO 3 — número de soluções (posto de A vs. posto de [A|b])")
print("=" * 60)


def classificar(A, b):
    """Classifica o sistema comparando posto(A), posto([A|b]) e n."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    rA = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.hstack([A, b]))
    n = A.shape[1]
    if rA < rAb:
        return rA, rAb, "nenhuma solução (sistema impossível)"
    if rA == n:
        return rA, rAb, "solução única"
    return rA, rAb, "infinitas soluções"


casos = {
    "3a) x+y=3, 2x+2y=6": ([[1, 1], [2, 2]], [3, 6]),
    "3b) x+y=3, 2x+2y=7": ([[1, 1], [2, 2]], [3, 7]),
    "3c) x+y=3, 2x-y=0":  ([[1, 1], [2, -1]], [3, 0]),
}
for nome, (A, b) in casos.items():
    rA, rAb, res = classificar(A, b)
    print(f"{nome}: posto(A)={rA}, posto([A|b])={rAb} -> {res}")

# RESPOSTAS CONCEITUAIS:
# 3a) L2 <- L2 - 2 L1 gera [0 0 | 0]: equação redundante, 1 variável livre
#     -> infinitas soluções (x = 3 - y, y livre).
# 3b) L2 <- L2 - 2 L1 gera [0 0 | 1], isto é, 0 = 1: contradição
#     -> nenhuma solução.
# 3c) L2 <- L2 - 2 L1 gera [0 -3 | -6] -> y = 2, x = 1: solução única.


# ---------------------------------------------------------------------------
# EXERCÍCIO 5b — Modelo de balanço térmico
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXERCÍCIO 5b — Balanço térmico")
print("=" * 60)

A5 = [[4, -1, 0],
      [-1, 4, -1],
      [0, -1, 3]]
b5 = [15, 10, 10]

T_gauss = gauss_sem_pivoteamento(A5, b5)
T_numpy = np.linalg.solve(np.array(A5, dtype=float), np.array(b5, dtype=float))

print("T (Gauss implementado):", T_gauss)
print("T (numpy.linalg.solve):", T_numpy)
print("Resíduo ||A T - b||:", np.linalg.norm(np.array(A5) @ T_gauss - np.array(b5)))

# INTERPRETAÇÃO:
# T1 = T2 = T3 = 5 (na unidade de temperatura do modelo). A matriz é
# diagonalmente dominante e simétrica definida positiva, portanto o sistema
# tem solução única e a eliminação de Gauss sem pivoteamento é estável
# (os pivôs 4, 15/4 e 41/15 são todos positivos).

EXERCÍCIO 4 — Gauss sem pivoteamento (sistema do exercício 2)
Matriz aumentada inicial:
 [[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]] 

L2 <- L2 - (-1.5000) * L1
L3 <- L3 - (-1.0000) * L1
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]] 

L3 <- L3 - (4.0000) * L2
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]] 

Solução (Gauss implementado): [ 2.  3. -1.]
Solução (numpy.linalg.solve): [ 2.  3. -1.]
Diferença máxima |x_gauss - x_numpy|: 5.551115123125783e-16
Resíduo ||A x - b||: 0.0

EXERCÍCIO 1 — forma matricial (verificação das soluções)
1a) A =
 [[ 2.  1.]
 [ 1. -3.]] 
    b = [ 7. -5.] 
    solução: [2.28571429 2.42857143]
1b) A =
 [[ 1.  2. -1.]
 [ 3. -1.  2.]
 [ 2.  1.  1.]] 
    b = [2. 9. 7.] 
    solução: [2. 1. 2.]

EXERCÍCIO 3 — número de soluções (posto de A vs. posto de [A|b])
3a) x+y=3, 2x+2y=6: posto(A)=1, posto([A|b])=1 -> infinitas soluções
3b) x+y=3, 2x+2y=7: posto(A)=1, posto([A|b])=2 -> nenhuma soluç

### Configuração Inicial e Função de Eliminação de Gauss (Exercício 4)

In [2]:
import numpy as np

def gauss_sem_pivoteamento(A, b, mostrar_passos=False):
    """
    Resolve A x = b por eliminação de Gauss sem pivoteamento.

    Parâmetros
    ----------
    A : matriz n x n dos coeficientes
    b : vetor independente de tamanho n
    mostrar_passos : se True, imprime a matriz aumentada a cada etapa

    Retorna
    -------
    x : vetor solução (np.ndarray)
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    # 1) Forma a matriz aumentada [A | b]
    M = np.hstack([A, b.reshape(-1, 1)])
    if mostrar_passos:
        print("Matriz aumentada inicial:\n", M, "\n")

    # 2) Eliminação: zera os elementos abaixo de cada pivô M[k, k]
    for k in range(n - 1):
        pivo = M[k, k]
        if abs(pivo) < 1e-12:
            # Sem pivoteamento, pivô nulo inviabiliza o método
            raise ZeroDivisionError(
                f"Pivô nulo na etapa {k + 1}: é necessário pivoteamento."
            )
        for i in range(k + 1, n):
            m = M[i, k] / pivo            # multiplicador m_ik
            M[i, k:] = M[i, k:] - m * M[k, k:]   # L_i <- L_i - m * L_k
            if mostrar_passos:
                print(f"L{i + 1} <- L{i + 1} - ({m:.4f}) * L{k + 1}")
        if mostrar_passos:
            print(M, "\n")

    # 3) Substituição regressiva (de baixo para cima)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - soma) / M[i, i]

    return x

### Teste do Exercício 4 (Sistema do Exercício 2)

In [3]:
print("=" * 60)
print("EXERCÍCIO 4 — Gauss sem pivoteamento (sistema do exercício 2)")
print("=" * 60)

A2 = [[2, 1, -1],
      [-3, -1, 2],
      [-2, 1, 2]]
b2 = [8, -11, -3]

x_gauss = gauss_sem_pivoteamento(A2, b2, mostrar_passos=True)
x_numpy = np.linalg.solve(np.array(A2, dtype=float), np.array(b2, dtype=float))

print("Solução (Gauss implementado):", x_gauss)
print("Solução (numpy.linalg.solve):", x_numpy)
print("Diferença máxima |x_gauss - x_numpy|:", np.max(np.abs(x_gauss - x_numpy)))
print("Resíduo ||A x - b||",
      np.linalg.norm(np.array(A2) @ x_gauss - np.array(b2)))

# INTERPRETAÇÃO:
# O método implementado retorna x = 2, y = 3, z = -1, idêntico (a menos de
# erro de arredondamento da ordem de 1e-16) ao resultado de numpy.linalg.solve,
# que usa fatoração LU com pivoteamento parcial. Como os pivôs deste sistema
# (2, 1/2, -1) são todos não nulos, o pivoteamento não era necessário aqui.
# Observação: sem pivoteamento o método falha quando algum pivô é zero e pode
# ser numericamente instável quando o pivô é muito pequeno.

EXERCÍCIO 4 — Gauss sem pivoteamento (sistema do exercício 2)
Matriz aumentada inicial:
 [[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]] 

L2 <- L2 - (-1.5000) * L1
L3 <- L3 - (-1.0000) * L1
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]] 

L3 <- L3 - (4.0000) * L2
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]] 

Solução (Gauss implementado): [ 2.  3. -1.]
Solução (numpy.linalg.solve): [ 2.  3. -1.]
Diferença máxima |x_gauss - x_numpy|: 5.551115123125783e-16
Resíduo ||A x - b|| 0.0


### Exercício 1 — Verificação de Soluções

In [4]:
print("=" * 60)
print("EXERCÍCIO 1 — forma matricial (verificação das soluções)")
print("=" * 60)

A1a = np.array([[2, 1], [1, -3]], dtype=float)
b1a = np.array([7, -5], dtype=float)
print("1a) A =\n", A1a, "\n    b =", b1a, "\n    solução:", np.linalg.solve(A1a, b1a))

A1b = np.array([[1, 2, -1], [3, -1, 2], [2, 1, 1]], dtype=float)
b1b = np.array([2, 9, 7], dtype=float)
print("1b) A =\n", A1b, "\n    b =", b1b, "\n    solução:", np.linalg.solve(A1b, b1b))

EXERCÍCIO 1 — forma matricial (verificação das soluções)
1a) A =
 [[ 2.  1.]
 [ 1. -3.]] 
    b = [ 7. -5.] 
    solução: [2.28571429 2.42857143]
1b) A =
 [[ 1.  2. -1.]
 [ 3. -1.  2.]
 [ 2.  1.  1.]] 
    b = [2. 9. 7.] 
    solução: [2. 1. 2.]


### Exercício 3 — Classificação de Sistemas

In [5]:
print("=" * 60)
print("EXERCÍCIO 3 — número de soluções (posto de A vs. posto de [A|b])")
print("=" * 60)

def classificar(A, b):
    """Classifica o sistema comparando posto(A), posto([A|b]) e n."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    rA = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.hstack([A, b]))
    n = A.shape[1]
    if rA < rAb:
        return rA, rAb, "nenhuma solução (sistema impossível)"
    if rA == n:
        return rA, rAb, "solução única"
    return rA, rAb, "infinitas soluções"

casos = {
    "3a) x+y=3, 2x+2y=6": ([[1, 1], [2, 2]], [3, 6]),
    "3b) x+y=3, 2x+2y=7": ([[1, 1], [2, 2]], [3, 7]),
    "3c) x+y=3, 2x-y=0":  ([[1, 1], [2, -1]], [3, 0]),
}
for nome, (A, b) in casos.items():
    rA, rAb, res = classificar(A, b)
    print(f"{nome}: posto(A)={rA}, posto([A|b])={rAb} -> {res}")

# RESPOSTAS CONCEITUAIS:
# 3a) L2 <- L2 - 2 L1 gera [0 0 | 0]: equação redundante, 1 variável livre
#     -> infinitas soluções (x = 3 - y, y livre).
# 3b) L2 <- L2 - 2 L1 gera [0 0 | 1], isto é, 0 = 1: contradição
#     -> nenhuma solução.
# 3c) L2 <- L2 - 2 L1 gera [0 -3 | -6] -> y = 2, x = 1: solução única.

EXERCÍCIO 3 — número de soluções (posto de A vs. posto de [A|b])
3a) x+y=3, 2x+2y=6: posto(A)=1, posto([A|b])=1 -> infinitas soluções
3b) x+y=3, 2x+2y=7: posto(A)=1, posto([A|b])=2 -> nenhuma solução (sistema impossível)
3c) x+y=3, 2x-y=0: posto(A)=2, posto([A|b])=2 -> solução única


### Exercício 5b — Modelo de Balanço Térmico

In [6]:
print("=" * 60)
print("EXERCÍCIO 5b — Balanço térmico")
print("=" * 60)

A5 = [[4, -1, 0],
      [-1, 4, -1],
      [0, -1, 3]]
b5 = [15, 10, 10]

T_gauss = gauss_sem_pivoteamento(A5, b5)
T_numpy = np.linalg.solve(np.array(A5, dtype=float), np.array(b5, dtype=float))

print("T (Gauss implementado):", T_gauss)
print("T (numpy.linalg.solve):", T_numpy)
print("Resíduo ||A T - b||", np.linalg.norm(np.array(A5) @ T_gauss - np.array(b5)))

# INTERPRETAÇÃO:
# T1 = T2 = T3 = 5 (na unidade de temperatura do modelo). A matriz é
# diagonalmente dominante e simétrica definida positiva, portanto o sistema
# tem solução única e a eliminação de Gauss sem pivoteamento é estável
# (os pivôs 4, 15/4 e 41/15 são todos positivos).

EXERCÍCIO 5b — Balanço térmico
T (Gauss implementado): [5. 5. 5.]
T (numpy.linalg.solve): [5. 5. 5.]
Resíduo ||A T - b|| 0.0
